In [39]:
import pandas as pd
import numpy as np
import uuid
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from loguru import logger

In [4]:
raw_data = pd.read_csv("../data/raw/hotel_bookings.csv")

### Train - Test Split

In [28]:
TRAIN_SIZE = round(raw_data.shape[0] * 0.8)
TEST_SIZE = raw_data.shape[0] - TRAIN_SIZE

In [29]:
train_indices = raw_data.sample(TRAIN_SIZE).index

In [30]:
test_indices = [i for i in raw_data.index if i not in train_indices]

In [31]:
train_raw_data = raw_data.iloc[train_indices]
test_raw_data = raw_data.iloc[test_indices]

In [33]:
raw_data

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,0.0,0,BB,BEL,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,0.0,0,BB,FRA,Online TA,TA/TO,0,0,0,E,E,0,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,0.0,0,BB,DEU,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [32]:
train_raw_data

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
66009,City Hotel,1,280,2017,April,15,12,2,6,2,1.0,0,BB,LUX,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,126.00,0,1,Canceled,2016-07-09
97040,City Hotel,0,277,2016,September,37,5,2,5,2,0.0,0,HB,NOR,Offline TA/TO,TA/TO,0,0,0,A,A,0,No Deposit,21.0,NaN,0,Transient-Party,89.14,0,1,Check-Out,2016-09-12
104813,City Hotel,0,60,2017,January,3,20,1,2,2,0.0,1,BB,FRA,Online TA,TA/TO,0,0,0,D,D,1,No Deposit,9.0,NaN,0,Transient,102.60,0,2,Check-Out,2017-01-23
97668,City Hotel,0,315,2016,September,38,17,1,1,2,0.0,0,BB,FRA,Groups,TA/TO,0,0,0,A,A,0,No Deposit,1.0,NaN,38,Transient-Party,65.00,0,1,Check-Out,2016-09-19
79631,City Hotel,0,5,2015,October,44,28,0,3,2,0.0,0,BB,NLD,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,9.0,NaN,0,Contract,136.00,0,1,Check-Out,2015-10-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101618,City Hotel,0,116,2016,November,46,10,2,3,2,0.0,0,BB,BEL,Direct,Direct,0,0,0,A,A,0,No Deposit,14.0,NaN,0,Transient,105.74,0,0,Check-Out,2016-11-15
12175,Resort Hotel,1,247,2017,June,24,16,2,5,2,0.0,0,BB,IRL,Direct,Direct,0,0,0,A,A,0,No Deposit,250.0,NaN,0,Transient,72.09,0,1,Canceled,2017-02-01
40565,City Hotel,1,130,2015,August,31,1,2,1,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient-Party,76.50,0,1,Canceled,2015-07-07
10662,Resort Hotel,1,182,2017,March,13,26,4,6,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,44.70,0,2,Canceled,2016-09-29


In [36]:
train_raw_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 95512 entries, 66009 to 1484
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           95512 non-null  object 
 1   is_canceled                     95512 non-null  int64  
 2   lead_time                       95512 non-null  int64  
 3   arrival_date_year               95512 non-null  int64  
 4   arrival_date_month              95512 non-null  object 
 5   arrival_date_week_number        95512 non-null  int64  
 6   arrival_date_day_of_month       95512 non-null  int64  
 7   stays_in_weekend_nights         95512 non-null  int64  
 8   stays_in_week_nights            95512 non-null  int64  
 9   adults                          95512 non-null  int64  
 10  children                        95509 non-null  float64
 11  babies                          95512 non-null  int64  
 12  meal                            95

In [37]:
train_raw_data.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,95512.000000,95512.000000,95512.000000,95512.000000,95512.000000,95512.000000,95512.000000,95512.000000,95509.000000,95512.000000,95512.000000,95512.000000,95512.000000,95512.000000,82520.000000,5370.000000,95512.000000,95512.000000,95512.000000,95512.000000
mean,0.369985,103.858007,2016.155258,27.185694,15.794958,0.925884,2.501874,1.857599,0.104325,0.008041,0.031525,0.083581,0.133554,0.222045,86.347588,189.251955,2.322577,101.827889,0.063416,0.571960
std,0.482803,106.840490,0.707841,13.617152,8.778432,0.999190,1.909622,0.588177,0.399833,0.099513,0.174732,0.794028,1.438184,0.659955,110.514921,131.212528,17.694108,48.103908,0.246105,0.793128
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.350000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,178.500000,0.000000,94.800000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,629.000000,2017.000000,53.000000,31.000000,18.000000,42.000000,55.000000,10.000000,10.000000,1.000000,26.000000,71.000000,21.000000,535.000000,543.000000,391.000000,510.000000,8.000000,5.000000


#### Encoding

In [41]:
train_raw_data["hotel"].value_counts()

hotel
City Hotel      63549
Resort Hotel    31963
Name: count, dtype: int64

In [47]:
encoder = OneHotEncoder() # primer encoder
encoded = encoder.fit_transform(train_raw_data[["hotel"]])
encoder.categories_[0]

array(['City Hotel', 'Resort Hotel'], dtype=object)

In [42]:
train_raw_data["market_segment"].value_counts()

market_segment
Online TA        45242
Offline TA/TO    19334
Groups           15794
Direct           10157
Corporate         4195
Complementary      596
Aviation           193
Undefined            1
Name: count, dtype: int64

In [54]:
train_raw_data["lead_time"]

66009     280
97040     277
104813     60
97668     315
79631       5
         ... 
101618    116
12175     247
40565     130
10662     182
1484       68
Name: lead_time, Length: 95512, dtype: int64

In [79]:
class FeatureEngineeringProcessor:
    def __init__(self, raw_data: pd.DataFrame, pipeline_name: str) -> None:
        self.raw_data = raw_data
        self.pipeline_name = pipeline_name

    def impute_scale(self, n_components: int = 2) -> pd.DataFrame:
        """Pipeline que imputa variables numéricas y luego las escala,
        para finalmente Aplicar PCA y quedarse con N componentes principales"""
        numeric_cols = [
            "lead_time",
            "children",
            "adults",
            "babies",
            "adr"
        ]
        pipe = Pipeline(
            steps = [
                ("imputer_mean", SimpleImputer(strategy = "mean")),
                ("std_scaling", StandardScaler()),
                ("pca", PCA(n_components = n_components))
            ]
        )
        return pd.DataFrame(
            pipe.fit_transform(self.raw_data[numeric_cols]),
            columns = ["pc1", "pc2"]
        )

    def encode_categoricals(self) -> pd.DataFrame:
        encoded_vars = []
        for var in ["hotel", "market_segment", "reserved_room_type"]:
            logger.info(f"Codificando con OHE {var}")
            encoder = OneHotEncoder()
            encoded = encoder.fit_transform(self.raw_data[[var]]).toarray()
            cols = [f"{var}_{col}" for col in encoder.categories_[0]]
            _dataframe = pd.DataFrame(
                encoded,
                columns = cols
            )
            encoded_vars.append(_dataframe)
        return pd.concat(encoded_vars, axis = 1)
    
    def run(self) -> pd.DataFrame:
        # aca pondremos nuestro código
        logger.info(f"Inicializando Pipeline {self.pipeline_name}")
        
        categorical = self.encode_categoricals()
        numerics = self.impute_scale()

        modeling_dataset = pd.concat([categorical, numerics], axis = 1)
        # Dataset Previo al pipeline
        pipe = Pipeline(
            steps = [
                ("feature_selection", VarianceThreshold()),
                ("scaling_robust", RobustScaler())
            ]
        )
        
        return pd.DataFrame(
            pipe.fit_transform(modeling_dataset),
            columns = modeling_dataset.columns # si el VarianceThreshold elimina, ten cuidado
        )

In [82]:
train_processor = FeatureEngineeringProcessor(
    raw_data= train_raw_data,
    pipeline_name= "DSRP MLE2 Feature Engineering - TRAIN"
)
train_processor.run()

2026-02-15 07:15:59.785 | INFO     | __main__:run:44 - Inicializando Pipeline DSRP MLE2 Feature Engineering - TRAIN
2026-02-15 07:15:59.786 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE hotel
2026-02-15 07:15:59.803 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE market_segment
2026-02-15 07:15:59.816 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE reserved_room_type


,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,reserved_room_type_A,reserved_room_type_B,reserved_room_type_C,reserved_room_type_D,reserved_room_type_E,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,reserved_room_type_P,pc1,pc2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.847034,0.815314
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.089553,1.448746
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.162679,-1.184098
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.446109,1.714808
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.691145,-0.342293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95507,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.215887,0.398638
95508,-1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.315994,1.271213
95509,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.201688,0.514983
95510,-1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.672252,0.877957


In [83]:
test_processor = FeatureEngineeringProcessor(
    raw_data= test_raw_data,
    pipeline_name= "DSRP MLE2 Feature Engineering - TEST"
)
test_processor.run()

2026-02-15 07:16:10.149 | INFO     | __main__:run:44 - Inicializando Pipeline DSRP MLE2 Feature Engineering - TEST
2026-02-15 07:16:10.150 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE hotel
2026-02-15 07:16:10.157 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE market_segment
2026-02-15 07:16:10.161 | INFO     | __main__:encode_categoricals:31 - Codificando con OHE reserved_room_type


,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,reserved_room_type_A,reserved_room_type_B,reserved_room_type_C,reserved_room_type_D,reserved_room_type_E,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,reserved_room_type_P,pc1,pc2
0,-1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.279465,1.965921
1,-1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.326891,4.466729
2,-1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.141889,-0.253324
3,-1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.270485,-0.355058
4,-1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.121303,0.090014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23873,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.098300,-0.136923
23874,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.692745,1.466466
23875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.741137,-0.125605
23876,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.981503,-0.213599
